### Track C - Integrate Pybullet simulation with Pytorch model

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, random_split
import matplotlib.pyplot as plt
import numpy as np

import math
import time
from pathlib import Path
from PIL import Image

import numpy as np
import pybullet as p
import pybullet_data

### C1. Simplify the task drastically

1. Reach task

In [8]:
SEED=5
torch.manual_seed(SEED)
np.random.seed(SEED)

p.disconnect()
physicsClient = p.connect(p.GUI)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.setGravity(0, 0, -9.81)

# upload the franka arm.
planeID = p.loadURDF("plane.urdf")
franka_arm = p.loadURDF("franka_panda/panda.urdf",
                        basePosition=[0,0,0],
                        useFixedBase=True)

# create a sphere
radius = 0.1
visual_shape = p.createVisualShape(shapeType=p.GEOM_SPHERE,
                                radius=radius,
                                rgbaColor=[0, 1, 0, 1])

collision_shape = p.createCollisionShape(shapeType=p.GEOM_SPHERE,
                                        radius=radius)
                                        
sphere_id = p.createMultiBody(baseMass=0,
                              baseCollisionShapeIndex=collision_shape,
                              baseVisualShapeIndex=visual_shape,
                              basePosition=[1,1,1])

numActiveThreads = 0
stopping threads
Thread with taskId 0 exiting
Thread TERMINATED
destroy semaphore
semaphore destroyed
destroy main semaphore
main semaphore destroyed
finished
numActiveThreads = 0
btShutDownExampleBrowser stopping threads
Thread with taskId 0 exiting
Thread TERMINATED
destroy semaphore
semaphore destroyed
destroy main semaphore
main semaphore destroyed
startThreads creating 1 threads.
starting thread 0
started thread 0 
argc=2
argv[0] = --unused
argv[1] = --start_demo_name=Physics Server
ExampleBrowserThreadFunc started
X11 functions dynamically loaded using dlopen/dlsym OK!
X11 functions dynamically loaded using dlopen/dlsym OK!
Creating context
Created GL 3.3 context
Direct GLX rendering context obtained
Making context current
GL_VENDOR=Intel
GL_RENDERER=Mesa Intel(R) Graphics (MTL)
GL_VERSION=4.6 (Core Profile) Mesa 23.2.1-1ubuntu3.1~22.04.4
GL_SHADING_LANGUAGE_VERSION=4.60
pthread_getconcurrency()=0
Version = 4.6 (Core Profile) Mesa 23.2.1-1ubuntu3.1~22.04.4
Ve

In [11]:
def _get_end_effector(robotId):
    end_effector_index = None

    for i in range(p.getNumJoints(robotId)):
        link_name = p.getJointInfo(robotId, i)[12].decode("utf-8")

        if link_name == "panda_hand":
            end_effector_index = i
            break
    return end_effector_index

PANDA_ARM_JOINTS = [0, 1, 2, 3, 4, 5, 6]
PANDA_FINGER_JOINTS = [9, 10]
END_EFFECTOR = _get_end_effector(robotId = franka_arm)


class MiniArmEnv():
    def __init__(self, robotId, sphereId, max_steps=300):
        self.robotId = robotId
        self.sphereId = sphereId
        self.PANDA_ARM_JOINTS = PANDA_ARM_JOINTS
        self.PANDA_FINGER_JOINTS = PANDA_FINGER_JOINTS
        self.end_effector = END_EFFECTOR
        self.max_steps = max_steps
        self.physics_engine_update = 240 # 240 Hz.
        self.success_threshold = 0.5
        self._step_count = 0
        self.reset()
    
    def reset(self):
        home_positions = [0, -0.785, 0, -2.356, 0, 1.571, 0.785]
        finger_home = [0.04, 0.04]  # open gripper

        for j, angle in zip(self.PANDA_ARM_JOINTS, home_positions):
            p.resetJointState(self.robotId, j, angle, targetVelocity=0)

        for j, angle in zip(self.PANDA_FINGER_JOINTS, finger_home):
            p.resetJointState(self.robotId, j, angle, targetVelocity=0)

        # reset sphere position
        p.resetBasePositionAndOrientation(bodyUniqueId=self.sphereId,
                                          posObj=[np.random.uniform(-0.4, 0.4),
                                                  np.random.uniform(-0.4, 0.4),
                                                  np.random.uniform(0.4, 0.4)],
                                          ornObj=[0.0, 0.0, 0.0, 1.0])
        
        self.sphereId_position = np.array(p.getBasePositionAndOrientation(self.sphereId)[0])
        p.stepSimulation()
    
    def step(self, action):
        
        self._apply_action(action)

        p.setTimeStep(1 / self.physics_engine_update)

        for _ in range(12):
            # TODO: add a break. If the EE reached the target position it stops. Use self._check_ee_close_sphere() every N steps.
            p.stepSimulation()

        self._step_count += 1        
                  
        obs = self._get_obs()
        reward = None
        done = self._is_done() or self._step_count >= self.max_steps
        info = None

        return obs, reward, done, info

    # def close(self):
    #     p.disconnect(self.client)

    # ── action application ───────────────────────────────────────────────────

    def _apply_action(self, action):
        self.action = action # (x,y,z) in meters (MKS system).

        # inverse kinematics
        joint_angles = p.calculateInverseKinematics(
            bodyUniqueId=self.robotId,
            endEffectorLinkIndex=self.end_effector,
            targetPosition=self.action,
            maxNumIterations=400,
            residualThreshold=1e-7
        )

        # move the arm
        p.setJointMotorControlArray(
            bodyIndex=self.robotId,
            jointIndices=self.PANDA_ARM_JOINTS + self.PANDA_FINGER_JOINTS,
            controlMode=p.POSITION_CONTROL,
            targetPositions = joint_angles,
            forces = [500]*len(self.PANDA_ARM_JOINTS + self.PANDA_FINGER_JOINTS)
        )


    # ── observations ─────────────────────────────────────────────────────────

    def _get_obs(self):
        position = p.getLinkState(self.robotId, self.end_effector)[4]
        return {
            "position": position
        }

    # def _get_camera_view(self):
    #     view_matrix = p.computeViewMatrix(cameraEyePosition=[1,0,2], 
    #                               cameraTargetPosition=[0,0,1],
    #                               cameraUpVector=[0,1,0.5]
    #                               )
    #     projection_matrix = p.computeProjectionMatrixFOV(fov=60,
    #                                                 aspect=1.0,
    #                                                 nearVal=0.1,
    #                                                 farVal=10
    #                                                 )
                                        
    #     width, height, rgb, depth, segmentation = p.getCameraImage(width=1200,height=1200,viewMatrix=view_matrix,projectionMatrix=projection_matrix)

    #     ### Save camera view.
    #     # # Convert RGB data to a NumPy array
    #     # rgb_array = np.array(rgb, dtype=np.uint8).reshape(height, width, 4)

    #     # # Remove alpha channel (RGBA -> RGB)
    #     # rgb_array = rgb_array[:, :, :3]

    #     # # Save image
    #     # image = Image.fromarray(rgb_array)
    #     # image.save("camera_view.png")
    
    # ── reward & termination ─────────────────────────────────────────────────
    def _check_position_success(self):
        return np.sqrt(np.dot(self._get_obs()["position"] - self.sphereId_position, self._get_obs()["position"] - self.sphereId_position)) < self.success_threshold

    def _is_done(self):
        if self._check_position_success():
            print(f"End-effector is close (< {self.success_threshold}m) to sphere position")
        return self._check_position_success()


In [ ]:
arm_env = MiniArmEnv(robotId=franka_arm, sphereId=sphere_id)
delta_position = np.array([0,0.1,0.03])
arm_env.step(action=arm_env.sphereId_position + delta_position)

End-effector is close (< 0.5m) to sphere position


({'position': (0.09698016941547394, -0.17194834351539612, 0.5208860635757446)},
 None,
 True,
 None)

: 

### C2. Oracle script

TODO: with ORACLE() you generate the action (the one that goes inside MiniArmEnv().step(action)) and the reset of the position of the object (sphere)

In [ ]:



class Oracle():
    def __init__(self):

    ## GENERATE THE ACTION THAT GOES INSIDE MiniArmEnv().step(action)

    def reset(self):
        
    
def collect_demos(n_demos, domain, out_path):
    while not done:
        # # override phase in obs with oracle's phase counter
        # obs["phase"] = oracle.advance_phase(obs)
        # action = oracle.act(obs)

        # ep_rgb.append(obs["rgb"])
        # ep_depth.append(obs["depth"])
        # ep_proprio.append(obs["proprio"])
        # ep_force.append(obs["force"])
        # ep_action.append(action)
        # ep_phase.append(obs["phase"])

        obs, _, done, info = env.step(action)